# LAB 9 — Statistical Analysis
## Parts 2 & 3: Pairwise Comparisons + Multiple Comparisons Correction

**TLDR (plain English):** We ran 90 days of ads across 7 channels and collected daily data. Now we need to know: are the performance differences between channels *real*, or did we just get lucky/unlucky on some days? We'll use proper hypothesis tests to find out — and then apply a 'false positive filter' because when you run lots of tests at once, some will look significant just by chance.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from itertools import combinations
from scipy.stats import ttest_ind, fisher_exact

try:
    from scipy.stats import false_discovery_control
    HAS_FDR = True
    print("scipy FDR available")
except ImportError:
    HAS_FDR = False
    print("scipy < 1.11 — using manual BH")

import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11})

OUTPUT_DIR = 'outputs/'
DATA_PATH  = 'data/marketing_data.csv'

df = pd.read_csv(DATA_PATH, parse_dates=['date'])
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Daily CPA (only rows with conversions > 0)
df['daily_cpa'] = df.apply(lambda r: r['cost']/r['conversions'] if r['conversions'] > 0 else np.nan, axis=1)
df['daily_cvr'] = df['conversions'] / df['clicks'] * 100

# Exclude SEO from paid-metric comparisons (zero cost = undefined CPA/ROAS)
PAID_CHANNELS = [ch for ch in df['channel'].unique() if ch != 'SEO_Organic']
ALL_CHANNELS  = sorted(df['channel'].unique())

print(f"Loaded {len(df):,} rows | All channels: {ALL_CHANNELS}")
print(f"Paid channels for CPA analysis: {PAID_CHANNELS}")

---
## Part 2 — Step 4: Pairwise t-tests on Daily CPA

**TLDR:** A t-test answers: 'Are the average CPAs of Channel A and Channel B genuinely different, or could that gap have appeared just from random day-to-day noise?' We run this for every possible pair of channels.

**Cohen's d** tells us *how big* the difference is — not just whether it exists. A statistically significant but tiny difference may not be worth acting on.

In [ ]:
def cohens_d(a, b):
    """Pooled-SD Cohen's d effect size."""
    pooled_std = np.sqrt((np.var(a, ddof=1) + np.var(b, ddof=1)) / 2)
    return (np.mean(b) - np.mean(a)) / pooled_std if pooled_std > 0 else 0.0

def interpret_d(d):
    d = abs(d)
    if d < 0.2:  return 'negligible'
    if d < 0.5:  return 'small'
    if d < 0.8:  return 'medium'
    return 'large'

cpa_results = []

for ch_a, ch_b in combinations(PAID_CHANNELS, 2):
    vals_a = df[(df['channel']==ch_a)]['daily_cpa'].dropna().values
    vals_b = df[(df['channel']==ch_b)]['daily_cpa'].dropna().values
    if len(vals_a) < 5 or len(vals_b) < 5:
        continue

    t_stat, p_val = ttest_ind(vals_a, vals_b, equal_var=False)  # Welch's t-test
    mean_a, mean_b = np.mean(vals_a), np.mean(vals_b)
    diff    = mean_b - mean_a
    pct_diff = (diff / mean_a) * 100 if mean_a != 0 else np.nan
    d        = cohens_d(vals_a, vals_b)

    cpa_results.append({
        'channel_a':        ch_a,
        'channel_b':        ch_b,
        'mean_cpa_a':       round(mean_a, 2),
        'mean_cpa_b':       round(mean_b, 2),
        'difference':       round(diff, 2),
        'pct_difference':   round(pct_diff, 1),
        't_statistic':      round(t_stat, 3),
        'p_value':          p_val,
        'cohens_d':         round(d, 3),
        'effect_size':      interpret_d(d),
        'significant_raw':  p_val < 0.05,
    })

cpa_df = pd.DataFrame(cpa_results)
n_sig  = cpa_df['significant_raw'].sum()

print(f"CPA pairwise comparisons: {len(cpa_df)} total, {n_sig} significant at α=0.05 (raw)")
print()
pd.set_option('display.float_format', '{:.4f}'.format)
cpa_df[['channel_a','channel_b','mean_cpa_a','mean_cpa_b','difference',
         'pct_difference','p_value','cohens_d','effect_size','significant_raw']].sort_values('p_value')

In [ ]:
# Heatmap of CPA p-values
fig, ax = plt.subplots(figsize=(9, 7))

pval_matrix = pd.DataFrame(1.0, index=PAID_CHANNELS, columns=PAID_CHANNELS)
for _, row in cpa_df.iterrows():
    pval_matrix.loc[row['channel_a'], row['channel_b']] = row['p_value']
    pval_matrix.loc[row['channel_b'], row['channel_a']] = row['p_value']

mask = np.eye(len(PAID_CHANNELS), dtype=bool)
sns.heatmap(
    pval_matrix, annot=True, fmt='.4f', cmap='RdYlGn_r',
    vmin=0, vmax=0.10, mask=mask, ax=ax,
    linewidths=0.5, cbar_kws={'label': 'p-value (red = significant)'}
)
ax.set_title('CPA Pairwise t-test p-values\n(red < 0.05 = significant difference)', fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'metric_comparison_heatmap.png', bbox_inches='tight')
plt.show()
print("Saved: metric_comparison_heatmap.png")

---
## Step 5: Fisher's Exact Test on Conversion Rates

**TLDR:** Fisher's exact test is the right tool for binary outcomes (converted / didn't convert). Unlike the t-test which compares daily averages, Fisher's works directly on counts: 'Channel A had X conversions out of Y clicks — is that rate meaningfully different from Channel B?'

This is more rigorous than comparing conversion rate averages because it respects the actual scale of the data.

In [ ]:
# Aggregate totals — clicks as 'total attempts', conversions as successes
conv_agg = df.groupby('channel').agg(
    conversions=('conversions', 'sum'),
    clicks=('clicks', 'sum')
).reset_index()
conv_agg['non_conversions'] = conv_agg['clicks'] - conv_agg['conversions']
conv_agg['rate_pct'] = conv_agg['conversions'] / conv_agg['clicks'] * 100

print("Conversion summary table:")
print(conv_agg[['channel','conversions','clicks','non_conversions','rate_pct']].sort_values('rate_pct', ascending=False).to_string(index=False))

In [ ]:
fisher_results = []

for ch_a, ch_b in combinations(ALL_CHANNELS, 2):
    row_a = conv_agg[conv_agg['channel']==ch_a].iloc[0]
    row_b = conv_agg[conv_agg['channel']==ch_b].iloc[0]

    table = [
        [row_a['conversions'], row_a['non_conversions']],
        [row_b['conversions'], row_b['non_conversions']],
    ]
    odds_ratio, p_val = fisher_exact(table, alternative='two-sided')

    rate_a = row_a['rate_pct']
    rate_b = row_b['rate_pct']
    diff   = rate_b - rate_a
    pct_diff = (diff / rate_a) * 100 if rate_a != 0 else np.nan

    fisher_results.append({
        'channel_a':        ch_a,
        'channel_b':        ch_b,
        'conversions_a':    int(row_a['conversions']),
        'conversions_b':    int(row_b['conversions']),
        'clicks_a':         int(row_a['clicks']),
        'clicks_b':         int(row_b['clicks']),
        'rate_a_pct':       round(rate_a, 3),
        'rate_b_pct':       round(rate_b, 3),
        'diff_pct_pts':     round(diff, 3),
        'pct_difference':   round(pct_diff, 1),
        'odds_ratio':       round(odds_ratio, 3),
        'p_value':          p_val,
        'significant_raw':  p_val < 0.05,
    })

fisher_df = pd.DataFrame(fisher_results)
n_sig_fisher = fisher_df['significant_raw'].sum()

print(f"Fisher's exact comparisons: {len(fisher_df)} total, {n_sig_fisher} significant at α=0.05 (raw)")
print()
fisher_df[['channel_a','channel_b','rate_a_pct','rate_b_pct','diff_pct_pts','odds_ratio','p_value','significant_raw']].sort_values('p_value')

In [ ]:
# Visualise conversion rates per channel
fig, ax = plt.subplots(figsize=(9, 5))
sorted_conv = conv_agg.sort_values('rate_pct', ascending=True)
colors = ['#2196F3' if ch != 'SEO_Organic' else '#4CAF50' for ch in sorted_conv['channel']]
bars = ax.barh(sorted_conv['channel'], sorted_conv['rate_pct'], color=colors, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, sorted_conv['rate_pct']):
    ax.text(bar.get_width() + 0.03, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}%', va='center', fontsize=10)
ax.set_xlabel('Conversion Rate (clicks → conversions) %')
ax.set_title('Conversion Rate by Channel', fontweight='bold')
ax.set_xlim(0, sorted_conv['rate_pct'].max() * 1.15)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'rate_comparison.png', bbox_inches='tight')
plt.show()
print("Saved: rate_comparison.png")

---
## Part 3 — Step 6: Multiple Comparisons Correction

**TLDR — The multiple comparisons problem in plain English:**  
If you flip a fair coin 20 times, you'll occasionally get 5 heads in a row — not because the coin is rigged, but because random stuff happens. If you run 21 statistical tests, you expect about 1 to look 'significant' just by chance. We tested 15 CPA pairs and 21 conversion rate pairs = 36 total. So ~1-2 of our 'significant' findings could be complete noise.

**Bonferroni** is the strict one — divide your significance threshold by the number of tests. Very conservative, almost no false positives, but misses real effects.  
**Benjamini-Hochberg (FDR)** is smarter — it controls the *proportion* of your significant findings that are false. Better balance for business decisions.

In [ ]:
alpha = 0.05
n_cpa      = len(cpa_df)
n_fisher   = len(fisher_df)
n_total    = n_cpa + n_fisher
expected_fp = round(n_total * alpha, 1)

print("=== MULTIPLE COMPARISONS PROBLEM ===")
print(f"CPA comparisons:              {n_cpa}")
print(f"Conversion rate comparisons:  {n_fisher}")
print(f"Total tests run:              {n_total}")
print(f"Expected false positives at α=0.05: ~{expected_fp}")
print(f"\nOf our {cpa_df['significant_raw'].sum() + fisher_df['significant_raw'].sum()} 'significant' raw results,")
print(f"some are likely noise. Corrections below will filter these out.")

In [ ]:
# --- Bonferroni correction ---
alpha_bonf_cpa    = alpha / n_cpa
alpha_bonf_fisher = alpha / n_fisher

cpa_df['significant_bonferroni']    = cpa_df['p_value']    < alpha_bonf_cpa
fisher_df['significant_bonferroni'] = fisher_df['p_value'] < alpha_bonf_fisher

print(f"Bonferroni α for CPA:    {alpha_bonf_cpa:.5f}   → {cpa_df['significant_bonferroni'].sum()} significant")
print(f"Bonferroni α for Fisher: {alpha_bonf_fisher:.5f} → {fisher_df['significant_bonferroni'].sum()} significant")

print("\nCPA pairs surviving Bonferroni:")
bonf_cpa = cpa_df[cpa_df['significant_bonferroni']][['channel_a','channel_b','mean_cpa_a','mean_cpa_b','p_value','cohens_d']]
print(bonf_cpa.to_string(index=False) if len(bonf_cpa) else "  None")

print("\nConversion rate pairs surviving Bonferroni:")
bonf_f = fisher_df[fisher_df['significant_bonferroni']][['channel_a','channel_b','rate_a_pct','rate_b_pct','p_value','odds_ratio']]
print(bonf_f.to_string(index=False) if len(bonf_f) else "  None")

In [ ]:
# --- Benjamini-Hochberg FDR correction ---
def manual_bh(p_values, alpha=0.05):
    """Manual BH adjustment — returns adjusted p-values."""
    n  = len(p_values)
    order = np.argsort(p_values)
    adj  = np.empty(n)
    for i, orig_idx in enumerate(order):
        adj[orig_idx] = p_values[orig_idx] * n / (i + 1)
    # Enforce monotonicity (cummin from end)
    adj_sorted = adj[order]
    for i in range(n - 2, -1, -1):
        adj_sorted[i] = min(adj_sorted[i], adj_sorted[i + 1])
    adj[order] = adj_sorted
    return np.minimum(adj, 1.0)

if HAS_FDR:
    cpa_pvals_adj    = false_discovery_control(cpa_df['p_value'].values,    method='bh')
    fisher_pvals_adj = false_discovery_control(fisher_df['p_value'].values, method='bh')
else:
    cpa_pvals_adj    = manual_bh(cpa_df['p_value'].values)
    fisher_pvals_adj = manual_bh(fisher_df['p_value'].values)

cpa_df['p_value_fdr']    = cpa_pvals_adj
cpa_df['significant_fdr'] = cpa_df['p_value_fdr'] < alpha

fisher_df['p_value_fdr']    = fisher_pvals_adj
fisher_df['significant_fdr'] = fisher_df['p_value_fdr'] < alpha

print(f"FDR-adjusted CPA significant pairs:    {cpa_df['significant_fdr'].sum()} / {n_cpa}")
print(f"FDR-adjusted Fisher significant pairs: {fisher_df['significant_fdr'].sum()} / {n_fisher}")

print("\nCPA pairs surviving FDR:")
fdr_cpa = cpa_df[cpa_df['significant_fdr']][['channel_a','channel_b','mean_cpa_a','mean_cpa_b','p_value','p_value_fdr','cohens_d','effect_size']]
print(fdr_cpa.to_string(index=False) if len(fdr_cpa) else "  None")

print("\nConversion rate pairs surviving FDR:")
fdr_f = fisher_df[fisher_df['significant_fdr']][['channel_a','channel_b','rate_a_pct','rate_b_pct','p_value','p_value_fdr','odds_ratio']]
print(fdr_f.to_string(index=False) if len(fdr_f) else "  None")

In [ ]:
# Summary comparison of correction methods
summary_correction = pd.DataFrame({
    'Method': ['Uncorrected (α=0.05)', 'Bonferroni', 'FDR (Benjamini-Hochberg)'],
    'CPA significant':          [cpa_df['significant_raw'].sum(),
                                  cpa_df['significant_bonferroni'].sum(),
                                  cpa_df['significant_fdr'].sum()],
    'Conv Rate significant':    [fisher_df['significant_raw'].sum(),
                                  fisher_df['significant_bonferroni'].sum(),
                                  fisher_df['significant_fdr'].sum()],
})
summary_correction['Total significant'] = summary_correction['CPA significant'] + summary_correction['Conv Rate significant']
print(summary_correction.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4))
x       = np.arange(3)
width   = 0.28
b1 = ax.bar(x - width, summary_correction['CPA significant'],       width, label='CPA',           color='steelblue',  alpha=0.85)
b2 = ax.bar(x,          summary_correction['Conv Rate significant'], width, label='Conv Rate',     color='seagreen',   alpha=0.85)
b3 = ax.bar(x + width,  summary_correction['Total significant'],    width, label='Total',          color='tomato',     alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(summary_correction['Method'])
ax.set_ylabel('Number of Significant Pairs')
ax.set_title('Effect of Multiple Comparisons Correction on Significant Findings', fontweight='bold')
ax.legend()
for bar in [*b1, *b2, *b3]:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.05, str(int(h)), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'correction_comparison.png', bbox_inches='tight')
plt.show()
print("Saved: correction_comparison.png")

In [ ]:
# Save results for downstream use by power_analysis.py and memo generation
cpa_df.to_csv(OUTPUT_DIR + 'cpa_comparisons.csv', index=False)
fisher_df.to_csv(OUTPUT_DIR + 'fisher_comparisons.csv', index=False)
print("Results saved to outputs/")